# 채용 공고 데이터 hire_query 생성 노트북

이 노트북은 아래 두 CSV 파일을 읽어 OpenAI `gpt-4o-mini` 모델로 채용 상황을 자연어 요약하고, `condition`, `hire_query` 컬럼만 포함한 CSV를 생성합니다.

- `jobpreffered_preprocessed.csv` → `preffered_query.csv`
- `qualification_preprocessed.csv` → `qualified_query.csv`

환경변수는 `OPENAI_API_KEY`를 우선 사용하고, 없으면 `OPENAI_API`를 사용합니다.


In [26]:
# 1. 라이브러리 import 및 API Key 설정

from google.colab import userdata



import time
import ast
from pathlib import Path

import pandas as pd
from openai import OpenAI


# 환경변수 우선순위:
# 1) OPENAI_API_KEY
# 2) OPENAI_API
api_key = userdata.get("OPENAI_API_KEY")

if not api_key:
    raise RuntimeError(
        "OpenAI API Key가 없습니다. 환경변수 OPENAI_API_KEY 또는 OPENAI_API에 API Key를 설정해주세요."
    )

client = OpenAI(api_key=api_key)

MODEL_NAME = "gpt-4o-mini"

# API 비용 확인용 설정
# None이면 전체 행 처리, 2로 바꾸면 각 파일에서 앞 2행만 샘플 처리합니다.
MAX_ROWS = None

print("OpenAI client 설정 완료")
print(f"사용 모델: {MODEL_NAME}")
print(f"처리 행 제한: {'전체 행' if MAX_ROWS is None else str(MAX_ROWS) + '행'}")


OpenAI client 설정 완료
사용 모델: gpt-4o-mini
처리 행 제한: 전체 행


In [27]:
# 2. CSV 로드 및 컬럼 검증 함수

REQUIRED_COLUMNS = ["title", "job_category", "skill_keywords", "condition"]


def read_csv_with_fallback(path: str) -> pd.DataFrame:
    """여러 인코딩을 순차적으로 시도해 CSV를 읽습니다."""
    encodings = ["utf-8-sig", "utf-8", "cp949"]
    last_error = None

    for encoding in encodings:
        try:
            df = pd.read_csv(path, encoding=encoding)
            print(f"[로드 성공] {path} / encoding={encoding} / rows={len(df):,}")
            return df
        except UnicodeDecodeError as e:
            last_error = e
        except FileNotFoundError:
            raise FileNotFoundError(f"파일을 찾을 수 없습니다: {path}")

    raise UnicodeDecodeError(
        "unknown",
        b"",
        0,
        1,
        f"CSV 인코딩을 확인해주세요. 시도한 인코딩: {encodings}. 마지막 오류: {last_error}",
    )


def validate_columns(df: pd.DataFrame, required_columns=None) -> None:
    """필수 컬럼 존재 여부를 검증합니다."""
    if required_columns is None:
        required_columns = REQUIRED_COLUMNS

    missing = [col for col in required_columns if col not in df.columns]

    if missing:
        raise ValueError(
            "필수 컬럼이 누락되었습니다. "
            f"누락 컬럼: {missing} / 현재 컬럼: {list(df.columns)}"
        )


def load_and_prepare_csv(path: str, max_rows=None) -> pd.DataFrame:
    """CSV 로드, 필수 컬럼 검증, 결측치 처리, 선택적 행 제한을 수행합니다."""
    df = read_csv_with_fallback(path)
    validate_columns(df)

    # 필요한 컬럼만 사용
    df = df[REQUIRED_COLUMNS].copy()

    # 결측치는 빈 문자열로 처리
    df = df.fillna("")

    # API 비용 확인 및 샘플 실행용 옵션
    if max_rows is not None:
        df = df.head(max_rows).copy()
        print(f"[샘플 모드] 앞 {max_rows}행만 처리합니다.")

    print(f"[처리 대상] {path}: {len(df):,}행")
    return df


In [28]:
# 3. OpenAI 요약 함수

def normalize_skill_keywords(value) -> str:
    """skill_keywords 값이 리스트 문자열, 쉼표 구분 문자열, 일반 문자열이어도 자연스럽게 정리합니다."""
    if pd.isna(value):
        return ""

    text = str(value).strip()

    if not text:
        return ""

    # 리스트 문자열 형태 예: "['Python', 'Django', 'AWS']"
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (list, tuple, set)):
            return ", ".join(str(item).strip() for item in parsed if str(item).strip())
    except (ValueError, SyntaxError):
        pass

    # 일반 문자열 또는 쉼표 구분 문자열은 그대로 사용
    return text


def build_prompt(title: str, job_category: str, skill_keywords: str) -> str:
    """모델에 전달할 프롬프트를 구성합니다."""
    skill_keywords = normalize_skill_keywords(skill_keywords)

    return f"""
다음 채용 공고 데이터를 바탕으로, 이 공고가 어떤 기술과 어떤 역할을 요구하는지 한국어 자연어로 요약해줘.

[입력 데이터]
- 공고 제목: {title}
- 직무 카테고리: {job_category}
- 기술 키워드: {skill_keywords}

[작성 조건]
- 1~2문장으로 작성
- 단순 키워드 나열 금지
- 어떤 기술 역량과 어떤 역할 수행을 요구하는지 드러나야 함
- 입력된 정보에 근거해서만 작성
- 채용 조건을 과장하거나 추측하지 말 것
- 합격/불합격 판단, 점수화, 순위화 표현 금지
- 출력은 요약 문장만 작성
""".strip()


def generate_hire_query(
    title: str,
    job_category: str,
    skill_keywords: str,
    max_retries: int = 3,
    sleep_seconds: float = 1.5,
) -> str:
    """OpenAI API를 호출해 hire_query 문장을 생성합니다."""
    prompt = build_prompt(title, job_category, skill_keywords)

    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                temperature=0.2,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "너는 채용 공고 데이터를 요약하는 HR 데이터 전처리 보조자다. "
                            "입력 데이터에 근거해 간결하고 중립적인 한국어 문장으로 요약한다."
                        ),
                    },
                    {
                        "role": "user",
                        "content": prompt,
                    },
                ],
            )

            result = response.choices[0].message.content.strip()
            return result

        except Exception as e:
            print(f"[API 오류] attempt={attempt}/{max_retries} / error={e}")

            if attempt == max_retries:
                # 마지막 실패 시 빈 문자열 대신 오류 표식을 남겨 후처리 가능하게 함
                return f"[ERROR] hire_query 생성 실패: {e}"

            time.sleep(sleep_seconds * attempt)


In [29]:
# 4. 파일 처리 함수

def process_file(input_path: str, output_path: str, max_rows=None) -> pd.DataFrame:
    """입력 CSV를 처리해 condition, hire_query 컬럼만 포함한 출력 CSV를 생성합니다."""
    print("=" * 80)
    print(f"[시작] {input_path} → {output_path}")

    df = load_and_prepare_csv(input_path, max_rows=max_rows)

    print(f"[API 호출 예정 행 수] {len(df):,}행")
    print("주의: 각 행마다 OpenAI API 호출이 발생합니다.")
    print("-" * 80)

    hire_queries = []

    for idx, row in df.iterrows():
        title = str(row.get("title", ""))
        job_category = str(row.get("job_category", ""))
        skill_keywords = row.get("skill_keywords", "")

        print(f"[진행] {len(hire_queries) + 1:,}/{len(df):,} / title={title[:60]}")

        hire_query = generate_hire_query(
            title=title,
            job_category=job_category,
            skill_keywords=skill_keywords,
        )

        hire_queries.append(hire_query)

    result_df = pd.DataFrame({
        "condition": df["condition"].astype(str),
        "hire_query": hire_queries,
    })

    result_df.to_csv(output_path, index=False, encoding="utf-8-sig")

    print("-" * 80)
    print(f"[저장 완료] {output_path}")
    print(f"[원본 행 수] {len(df):,}")
    print(f"[결과 행 수] {len(result_df):,}")
    print("=" * 80)

    return result_df


In [33]:
# 5. 실행 셀

# MAX_ROWS = None이면 전체 처리
# 테스트로 2행만 처리하려면 위 설정 셀에서 MAX_ROWS = 2로 변경하세요.

preferred_result = process_file(
    input_path="jobpreferred_preprocessed.csv",
    output_path="preffered_query.csv",
    max_rows=MAX_ROWS,
)

qualified_result = process_file(
    input_path="qualification_preprocessed.csv",
    output_path="qualified_query.csv",
    max_rows=MAX_ROWS,
)


[시작] jobpreferred_preprocessed.csv → preffered_query.csv
[로드 성공] jobpreferred_preprocessed.csv / encoding=utf-8-sig / rows=389
[처리 대상] jobpreferred_preprocessed.csv: 389행
[API 호출 예정 행 수] 389행
주의: 각 행마다 OpenAI API 호출이 발생합니다.
--------------------------------------------------------------------------------
[진행] 1/389 / title=데이터 엔지니어 / 데이터 분석가
[진행] 2/389 / title=프론트엔드 개발자
[진행] 3/389 / title=프론트엔드 개발자
[진행] 4/389 / title=AI 엔지니어 / 머신러닝 엔지니어
[진행] 5/389 / title=AI 엔지니어 / 머신러닝 엔지니어
[진행] 6/389 / title=DevOps 엔지니어 / 인프라 엔지니어
[진행] 7/389 / title=DevOps 엔지니어 / 인프라 엔지니어
[진행] 8/389 / title=DevOps 엔지니어 / 인프라 엔지니어
[진행] 9/389 / title=프론트엔드 개발자
[진행] 10/389 / title=풀스택 개발자
[진행] 11/389 / title=풀스택 개발자
[진행] 12/389 / title=풀스택 개발자
[진행] 13/389 / title=보안 엔지니어
[진행] 14/389 / title=백엔드 개발자
[진행] 15/389 / title=백엔드 개발자
[진행] 16/389 / title=백엔드 개발자
[진행] 17/389 / title=백엔드 개발자
[진행] 18/389 / title=백엔드 개발자
[진행] 19/389 / title=백엔드 개발자
[진행] 20/389 / title=백엔드 개발자
[진행] 21/389 / title=백엔드 개발자
[진행] 22/389 / title=백엔드 개발자
[진

In [34]:
# 6. 결과 미리보기 셀

preffered_preview = pd.read_csv("preffered_query.csv", encoding="utf-8-sig")
qualified_preview = pd.read_csv("qualified_query.csv", encoding="utf-8-sig")

print("[preffered_query.csv 미리보기]")
display(preffered_preview.head())

print("[qualified_query.csv 미리보기]")
display(qualified_preview.head())


[preffered_query.csv 미리보기]


,condition,hire_query
0,postgreSQL DB 마이이그레이션 유경험자,이 채용 공고는 데이터 엔지니어 또는 데이터 분석가 역할을 수행할 인재를 찾고 있으...
1,"삼성PJT (전자, 반도체, 디스플레이 등) 경험있으면 우선검토",프론트엔드 개발자는 웹 애플리케이션의 사용자 인터페이스를 설계하고 구현하는 역할을 ...
2,"삼성PJT (전자, 반도체) 경험있으면 우선검토",프론트엔드 개발자는 웹 애플리케이션의 사용자 인터페이스를 개발하고 최적화하는 역할을...
3,"html, javascript, typescript 가능자","AI 엔지니어 및 머신러닝 엔지니어 직무에서 HTML, JavaScript, Typ..."
4,"우대 : 백엔드 가능자 ( 자바, Spring boot, oracle )","AI 엔지니어 및 머신러닝 엔지니어 직무에서 Java, Oracle, Spring,..."


[qualified_query.csv 미리보기]


,condition,hire_query
0,경력 9년 이상 ~ 15년 미만,"풀스택 개발자는 웹 애플리케이션의 프론트엔드와 백엔드 개발을 담당하며, 다양한 기술..."
1,학력 무관,"풀스택 개발자는 웹 애플리케이션의 프론트엔드와 백엔드 개발을 담당하며, 다양한 기술..."
2,경력 무관,데이터 엔지니어 및 데이터 분석가 직무에서는 데이터 처리 및 분석에 필요한 기술 역...
3,학력 무관,이 채용 공고는 데이터 엔지니어 및 데이터 분석가 역할을 수행할 인재를 찾고 있으며...
4,경력 10년 이상,데이터 엔지니어 및 데이터 분석가 직무에서는 데이터 처리 및 분석을 위한 기술 역량...


## 참고

- `MAX_ROWS = None`: 전체 행 처리
- `MAX_ROWS = 2`: 각 파일에서 앞 2행만 처리
- 출력 CSV는 `condition`, `hire_query` 두 컬럼만 포함합니다.
- API 호출 실패 행은 `[ERROR]`로 시작하는 문자열이 들어가므로, 실행 후 해당 행만 필터링해 재처리할 수 있습니다.


In [32]:
!dir

jobpreferred_preprocessed.csv  qualification_preprocessed.csv  sample_data
